# Phase 1C-2: Pre-game DNN 구조 선택

**목적**: 동일한 2025년 정규시즌 2,430경기와 기존 Phase 1C의 **시간순 데이터 분할**을 유지하면서, DNN의 은닉층 크기·Dropout·L2만 소폭 변경해 성능을 검증합니다.

- **모델 선택은 8월 Validation 자료로만 수행**합니다. 9월 Test는 모델 선정 후 **한 번만 평가**합니다.
- 선택 기준: **Validation Brier score 최소화** (동점에 가까울 때 Validation AUC 참고). 정확도도 함께 확인하지만 테스트 정확도만 높이는 임의 선택은 하지 않습니다.
- 투수 결측치에 대한 학습셋 중앙값 대체와 입력 피처 22개를 그대로 사용합니다.

**입력:** `phase1c_output.zip`. **출력:** `phase1c2_output.zip`.

In [ ]:
import sys, subprocess, importlib.util
for package, module in [('tensorflow','tensorflow'),('scikit-learn','sklearn'),('pandas','pandas'),('matplotlib','matplotlib')]:
    if importlib.util.find_spec(module) is None:
        subprocess.check_call([sys.executable,'-m','pip','install','-q',package])
import io, json, zipfile, os, random, shutil
from pathlib import Path
import numpy as np, pandas as pd
import tensorflow as tf
import matplotlib.pyplot as plt
from google.colab import files
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, roc_auc_score, brier_score_loss, log_loss
print('TensorFlow:',tf.__version__)

In [ ]:
uploaded=files.upload()
assert len(uploaded)==1, 'phase1c_output.zip 한 개만 업로드하세요.'
name=next(iter(uploaded))
assert name.endswith('.zip')
with zipfile.ZipFile(io.BytesIO(uploaded[name])) as z:
    df=pd.read_csv(z.open('pregame_features_cleaned_2430.csv'),parse_dates=['date'])
    partitions=pd.read_csv(z.open('game_partitions.csv'))
    baseline_metrics=pd.read_csv(z.open('pregame_metrics.csv'))
    previous_audit=json.load(z.open('training_audit.json'))
    previous_export=json.load(z.open('pregame_model_export.json'))
FEATURES=previous_export['features']; assert len(FEATURES)==22
assert len(df)==2430 and df.gamePk.nunique()==2430
check=df[['gamePk','split']].merge(partitions[['gamePk','split']],on='gamePk',suffixes=('_df','_partition'))
assert len(check)==2430 and (check.split_df==check.split_partition).all()
train=df.split.eq('train').to_numpy(); val=df.split.eq('validation').to_numpy(); test=df.split.eq('test').to_numpy()
assert [int(train.sum()),int(val.sum()),int(test.sum())]==[1679,377,374]
assert df.groupby('date')['split'].nunique().max()==1
assert df.loc[train,'date'].max() < df.loc[val,'date'].min() < df.loc[test,'date'].min()
assert not df[FEATURES].isna().any().any()
print('Samples:',int(train.sum()),int(val.sum()),int(test.sum()),'Features:',len(FEATURES))
print('Test set is sealed until AFTER selection.')

## 1. 기존 분할·전처리 고정

Phase 1C에서 생성한 22개 변수를 재사용하고, 스케일러를 오직 학습셋으로 적합합니다. 분석 단위와 변수는 유지되므로 이번 비교의 변화는 **DNN 구조와 정규화 설정**에 국한됩니다.

In [ ]:
Xraw=df[FEATURES].to_numpy(dtype='float64')
y=df.target.astype(int).to_numpy()
scaler=StandardScaler().fit(Xraw[train])
X=scaler.transform(Xraw).astype('float32')
Xt,Xv=X[train],X[val]
yt,yv=y[train],y[val]
SEED=2026
# 5개 후보: 1개 기존 구조 + 4개 소규모 변형
candidates=[
    {'name':'C0_reference','hidden':[128,64,32],'dropout':0.30,'l2':0.0010,'lr':0.0010},
    {'name':'C1_milder_regularization','hidden':[128,64,32],'dropout':0.15,'l2':0.0001,'lr':0.0010},
    {'name':'C2_compact','hidden':[64,32],'dropout':0.15,'l2':0.0001,'lr':0.0010},
    {'name':'C3_medium','hidden':[96,48],'dropout':0.20,'l2':0.0001,'lr':0.0005},
    {'name':'C4_compact_deep','hidden':[64,32,16],'dropout':0.10,'l2':0.0001,'lr':0.0005},
]
def build_dnn(config):
    from tensorflow.keras import regularizers
    inputs=tf.keras.Input(shape=(len(FEATURES),))
    x=inputs
    for i,units in enumerate(config['hidden']):
        x=tf.keras.layers.Dense(units,activation='relu',kernel_regularizer=regularizers.l2(config['l2']))(x)
        if i<len(config['hidden'])-1 and config['dropout']:
            x=tf.keras.layers.Dropout(config['dropout'])(x)
    outputs=tf.keras.layers.Dense(1,activation='sigmoid')(x)
    net=tf.keras.Model(inputs,outputs)
    net.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=config['lr']),loss='binary_crossentropy',metrics=['accuracy'])
    return net

def metrics(y_true,p):
    p=np.clip(np.asarray(p,dtype=float),1e-9,1-1e-9)
    return {'accuracy':float(accuracy_score(y_true,p>=0.5)),
            'auc':float(roc_auc_score(y_true,p)),
            'brier':float(brier_score_loss(y_true,p)),
            'log_loss':float(log_loss(y_true,p,labels=[0,1]))}

def ece(y_true,p,bins=10):
    p=np.asarray(p,dtype=float); edges=np.linspace(0,1,bins+1)
    bin_index=np.clip(np.digitize(p,edges,right=False)-1,0,bins-1)
    return float(sum(np.mean(bin_index==i)*abs(y_true[bin_index==i].mean()-p[bin_index==i].mean())
                    for i in range(bins) if np.any(bin_index==i)))


## 2. Validation에서 후보 모델 비교

모든 모델은 같은 학습·검증 데이터와 동일한 시드로 학습합니다. 9월 테스트 데이터는 이 셀에서 사용하지 않습니다.

In [ ]:
val_results=[]; fitted={}; histories={}
for config in candidates:
    tf.keras.backend.clear_session()
    random.seed(SEED);np.random.seed(SEED);tf.keras.utils.set_random_seed(SEED)
    try: tf.config.experimental.enable_op_determinism()
    except Exception: pass
    net=build_dnn(config)
    history=net.fit(Xt,yt,validation_data=(Xv,yv),epochs=180,batch_size=64,verbose=0,
                    callbacks=[tf.keras.callbacks.EarlyStopping(monitor='val_loss',patience=14,
                                                                  restore_best_weights=True)])
    pv=net.predict(Xv,verbose=0).ravel()
    record={**config,'parameters':int(net.count_params()),'best_epoch':int(np.argmin(history.history['val_loss'])+1),
            'epochs_completed':len(history.history['loss']),**metrics(yv,pv),'ECE_10_bins':ece(yv,pv)}
    val_results.append(record);fitted[config['name']]=net;histories[config['name']]=history.history
    print(config['name'], '| val acc',round(record['accuracy'],4), '| AUC',round(record['auc'],4),
          '| Brier',round(record['brier'],4),'| params',record['parameters'])
leaderboard=pd.DataFrame(val_results).sort_values(['brier','auc'],ascending=[True,False]).reset_index(drop=True)
print('\nVALIDATION LEADERBOARD (test not used):')
print(leaderboard[['name','parameters','best_epoch','accuracy','auc','brier','log_loss','ECE_10_bins']].to_string(index=False))
selected_name=str(leaderboard.loc[0,'name']); selected_model=fitted[selected_name]
print('\nSELECTED ON VALIDATION BRIER:',selected_name)


## 3. 최종 선정 모델의 9월 테스트 성능 확인

앞 셀에서 선정된 **하나의 모델만** 테스트셋에서 평가합니다. 테스트 성능이 기대보다 낮더라도 후보를 재선택하지 않습니다.

In [ ]:
Xe=X[test];ye=y[test]
p_test=selected_model.predict(Xe,verbose=0).ravel()
test_metrics=metrics(ye,p_test);test_metrics['ECE_10_bins']=ece(ye,p_test)
print('Selected model:',selected_name)
print('FINAL HELD-OUT TEST:',json.dumps(test_metrics,indent=2))
print('PREVIOUS PHASE 1C TEST (reference only; NOT used for model selection):')
print(baseline_metrics.to_string(index=False))
# Calibration and selected model loss curves for documentation
fig,axes=plt.subplots(1,2,figsize=(11,4))
h=histories[selected_name]
axes[0].plot(h['loss'],label='train');axes[0].plot(h['val_loss'],label='validation')
axes[0].legend();axes[0].set_title('Selected DNN: training history');axes[0].set_xlabel('Epoch')
from sklearn.calibration import calibration_curve
frac,mean=calibration_curve(ye,p_test,n_bins=8,strategy='quantile')
axes[1].plot([0,1],[0,1],linestyle='--',color='gray')
axes[1].plot(mean,frac,marker='o',label=selected_name)
axes[1].legend();axes[1].set_title('Selected DNN: test calibration')
axes[1].set_xlabel('Predicted win probability');axes[1].set_ylabel('Observed win rate')
plt.tight_layout();plt.savefig('phase1c2_diagnostic.png',dpi=160);plt.show()

## 4. 결과 패키지 내보내기

선정된 네트워크 가중치, 스케일러, 검증셋 후보 비교표, 테스트셋 예측과 지표를 저장합니다.

In [ ]:
outdir=Path('phase1c2_output');outdir.mkdir(exist_ok=True)
leaderboard.to_csv(outdir/'validation_candidate_leaderboard.csv',index=False)
pd.DataFrame([{'model':selected_name,**test_metrics}]).to_csv(outdir/'selected_test_metrics.csv',index=False)
pd.DataFrame({'gamePk':df.loc[test,'gamePk'].to_numpy(),
              'date':df.loc[test,'date'].dt.strftime('%Y-%m-%d').to_numpy(),
              'target':ye,'prob_selected_dnn':p_test}).to_csv(outdir/'selected_test_predictions.csv',index=False)
selected_model.save(outdir/'selected_pregame_model.keras')
with open(outdir/'selected_pregame_export.json','w') as f:
    json.dump({'features':FEATURES,'scaler':{'mean':scaler.mean_.tolist(),'scale':scaler.scale_.tolist()},
     'layers':[{'W':layer.get_weights()[0].tolist(),'b':layer.get_weights()[1].tolist(),
                'activation':layer.activation.__name__} for layer in selected_model.layers if isinstance(layer,tf.keras.layers.Dense)],
     'note':'No clipping; selected using validation Brier score only'},f)
shutil.copy('phase1c2_diagnostic.png',outdir/'phase1c2_diagnostic.png')
report={'source':'phase1c_output.zip','selection_rule':'lowest validation Brier score; tie-break validation AUC',
        'features':FEATURES,'fixed_split':previous_audit['chronological_split'],
        'candidates':candidates,'selected_model':selected_name,
        'selected_validation_metrics':leaderboard.iloc[0][['accuracy','auc','brier','log_loss','ECE_10_bins']].to_dict(),
        'test_metrics':test_metrics,
        'important_note':'No test-set feedback used to revise candidate models inside this notebook; further tuning on this same test set would compromise its holdout status.'}
with open(outdir/'experiment_audit.json','w') as f:json.dump(report,f,indent=2,ensure_ascii=False)
with zipfile.ZipFile('phase1c2_output.zip','w',zipfile.ZIP_DEFLATED) as z:
    for path in outdir.iterdir():z.write(path,arcname=path.name)
print('Created phase1c2_output.zip:',[(p.name,p.stat().st_size) for p in outdir.iterdir()])
files.download('phase1c2_output.zip')